In [ ]:
from enum import Enum
import numpy as np
import torch
from occhio import ToyModel, ModelGrid
from occhio.autoencoder import TiedLinearRelu, SynthAE
from occhio.distributions import (
    SparseUniform,
    HierarchyNode,
    SyntheticDataConfig,
    SyntheticDataModel,
)
from occhio.model_grid import Axis
from sae_lens import (
    StandardTrainingSAE,
    StandardTrainingSAEConfig,
    MatryoshkaBatchTopKTrainingSAE,
    MatryoshkaBatchTopKTrainingSAEConfig,
    BatchTopKTrainingSAE,
    BatchTopKTrainingSAEConfig,
    JumpReLUTrainingSAE,
    JumpReLUTrainingSAEConfig,
    MatchingPursuitTrainingSAE,
    MatchingPursuitTrainingSAEConfig,
)
from occhio.visualization_2 import RepresentationPlot
from occhio.visualization_2.core import CompositePlot
from occhio.visualization_2.plots import SAEClassificationMetricsPlot
from occhio.visualization_2.plots.sae_classification_metric import (
    SAEClassificationMetric,
    SAEClassificationMetricPlot,
    SAEMetricsComparisonPlot,
)

In [ ]:
DEVICE = "cpu"

In [ ]:
N_FEATURES = 2000
N_HIDDEN = 64

In [ ]:
# [2026-03-19 | OliverSieweke] TODO: Define this in occhio directly.
class AutoencoderType(Enum):
    TiedLinearRelu = TiedLinearRelu.__name__
    SynthAE = SynthAE.__name__

In [ ]:
def build_hierarchy(
    start_idx: int, n_roots: int, branching: int
) -> tuple[list[HierarchyNode], int]:
    """Build a forest of trees with mutual exclusion and parent-scaled magnitudes."""
    idx = start_idx
    roots = []
    for _ in range(n_roots):
        root_idx = idx
        idx += 1
        children = []
        for _ in range(branching):
            children.append(HierarchyNode(feature_idx=idx))
            idx += 1
        roots.append(
            HierarchyNode(
                feature_idx=root_idx,
                children=children,
                mutually_exclusive_children=True,
                parent_scaled=True,
            )
        )
    return roots, idx


hierarchy_roots, _ = build_hierarchy(start_idx=0, n_roots=8, branching=4)
config = SyntheticDataConfig(
    n_features=N_FEATURES,
    # Firing probabilities
    firing_prob_distribution="zipfian",
    p_max=0.4,
    p_min=0.5 / N_FEATURES,
    alpha=0.5,
    # Magnitudes — linear mean, folded-normal stdev
    mean_distribution="linear",
    mean_high=3.0,
    mean_low=1.0,
    std_distribution="folded_normal",
    folded_normal_mu=0.5,
    folded_normal_sigma=0.5,
    # Correlation
    correlation_rank=4,
    correlation_scale=0.1,
    # Hierarchy
    hierarchy=hierarchy_roots,
    compensate_probabilities=True,
    # Runtime
    device=DEVICE,
)

In [ ]:
def create_model(params):
    SEED = 199
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)

    match params["Autoencoder"]:
        case AutoencoderType.TiedLinearRelu:
            ae = TiedLinearRelu(
                N_FEATURES, N_HIDDEN, device=DEVICE, generator=generator
            )
        case AutoencoderType.SynthAE:
            ae = SynthAE(N_FEATURES, N_HIDDEN, device=DEVICE, generator=generator)

    return ToyModel(
        ae=ae,
        distribution=SyntheticDataModel(config, seed=SEED),
        device=DEVICE,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(
            label="Autoencoder",
            values=[AutoencoderType.SynthAE, AutoencoderType.TiedLinearRelu],
        ),
    ],
)

In [ ]:
grid.fit(15000)

<llm-progress-log-group></llm-progress-log-group>

In [ ]:
import importlib
import occhio.visualization_2
import occhio.visualization_2.core
import occhio.visualization_2.core.figure_wrappers
import occhio.visualization_2.core.base_plot
import occhio.visualization_2.core.composite_plot
import occhio.visualization_2.plots
import occhio.visualization_2.plots.feature_representation

# Reload in dependency order (leaves first)
importlib.reload(occhio.visualization_2.core.figure_wrappers)
importlib.reload(occhio.visualization_2.core.base_plot)
importlib.reload(occhio.visualization_2.core.composite_plot)
importlib.reload(occhio.visualization_2.core)
importlib.reload(occhio.visualization_2.plots.feature_representation)
importlib.reload(occhio.visualization_2.plots)
importlib.invalidate_caches()
importlib.reload(occhio.visualization_2)
# Re-import after reload to get updated references
from occhio.visualization_2.core.figure_wrappers import FigureProxy
from occhio.visualization_2 import plot_feature_representation

fig = plot_feature_representation(grid, height=1000)
fig.show()

In [ ]:
grid[0].superposition

In [ ]:
grid[1].superposition